# BM25 vs Dense 임베딩 기반 검색과 RRF 결합 성능 비교
> 서로 다른 검색방법의 결과가 상이할때 유용한 방법이다.

Reciprocal Rank Fusion(RRF)은 **여러 검색 결과의 순위를 합쳐서 하나의 최종 순위를 만드는 방법**이다.  
예를 들어, 키워드 검색 결과와 벡터 검색 결과처럼 서로 다른 방식으로 나온 문서 리스트가 있을 때, 각 문서가 각 리스트에서 몇 번째에 있는지(순위)를 이용해 점수를 매긴다.  

$$
\text{score}(d) = \sum_{q} \frac{1}{k + \text{rank}(q, d)}
$$

- $d$: 문서  
- $q$: 각 검색 결과 집합  
- $\text{rank}(q, d)$: 검색 결과 $q$에서 문서 $d$의 순위(1부터 시작)  
- $k$: 보통 60 정도로 쓰는 상수

즉, **여러 검색 결과에서 상위에 자주 등장하는 문서일수록 점수가 높아지고, 최종 순위에서 위로 올라간다**.  
이 방식은 각 검색 결과의 점수 범위가 달라도 상관없이, 순위만으로 융합하므로 간단하게 적용할 수 있다.

In [1]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [3]:
import pandas as pd

document_df = pd.read_csv('documents.csv')
queries_df = pd.read_csv('queries.csv')

queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=3;D4=1;D30=1
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3;D14=2;D26=1
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=3
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=3


In [4]:
# BM25 키워드 검색 모델
# - 문서를 토큰화해서 BM25 모델 생성
# - 검색어도 동일하게 토큰화하여 BM25 검색시 활용

from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi

kiwi = Kiwi()

def kiwi_tokenize(doc):
    return [token.form for token in kiwi.tokenize(doc)]

# 문서 내용 토큰화
tokenized_docs = [kiwi_tokenize(doc) for doc in document_df['content']]
bm25 = BM25Okapi(tokenized_docs)    # BM25 모델 생성 (토큰화된 문서)

# 질의문을 토큰화해서 BM25로 상위 문서 검색하는 함수
def bm25_search(query,top_k = 5):
    query_tokens = kiwi_tokenize(query)         # 검색어 토큰화
    scores = bm25.get_scores(query_tokens)      # 각 문서의 BM25 점수 계산

    # 점수 기준 내림차순 정렬한 인덱스
    ranked_idx = sorted(range(len(scores)), key=lambda i :scores[i],reverse=True)
    # 상위 top_k개의 doc_id 리스트
    retrieved_docs = [document_df['doc_id'].iloc[i] for i in ranked_idx[:top_k]]
    return retrieved_docs

bm25_search('제주도 관광 명소')

['D1', 'D2', 'D3', 'D4', 'D5']

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

# 벡터스토어 연결
vector_store = PineconeVectorStore(
    index_name= 'ir',   # 연결할 index명
    embedding= embeddings   # 사용할 임베딩 함수 (연결할 인덱스 차원과 임베딩 차원이 같아야 함
)

## 평가지표 계산

In [7]:
import numpy as np

def parse_relevant(relevant_str) -> dict[str,int]:
    """ 
    참조문서 답안 문자열을 dict로 파싱하는 함수

    relevant_str = "D1=3;D4=1;D30=1" -> {'D1' :3, 'D2':1, 'D30':1}
    """

    pairs = relevant_str.split(';')
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split('=')
        rel_dict[doc_id] = int(grade)

    return rel_dict

In [8]:
def compute_metrics(predicted, relevant_dict, k=5) -> tuple[float,float,float,float]:
    hits = sum([1 for doc in predicted[:k] if doc in relevant_dict])
    precision = hits /k     # 정밀도 = 맞춘갯수 / 전체갯수

    # Recall
    total_relevant = len(relevant_dict) # 전체 관련 문서 수
    recall = hits / total_relevant if total_relevant > 0 else 0 # 재현율 = 관련 문서 맞춘 수

    # MRR 예측치 중 첫 관련문서 순위 점수
    rr = 0  # 관련 문서 수
    for idx,doc in enumerate(predicted):
        if doc in relevant_dict:
            rr=1/(idx+1)
            break   # 첫번째 적중 rr반영 후 반복문 탈출

    # AP (MAP를 위한 사전 계산)
    num_correct = 0 # 현재까지의 적중 횟수
    precisions = [] # 적중시의 precision
    for idx, doc in enumerate(predicted[:k]):   # top_k 범위에서
        if doc in relevant_dict:
            num_correct+=1  # 적중시 1 누적
            precisions.append(num_correct/(idx+1))  # 현재 시점의 precision 기록
    ap = np.mean(precisions) if precisions else 0

    return precision, recall, rr, ap

# 여러 쿼리에 대한 평균 성능지표 계산하는 함수
def evaluate_all(method_results,queries_df,k=5):
    prec_list,rec_list,rr_list,ap_list = [],[],[],[]    # 지표별 결과 저장

    for idx, row in queries_df.iterrows():
        qid = row['query_id']   # 쿼리 id
        relevant_dict = parse_relevant(row['relevant_doc_ids'])  # 정답 dict 파싱
        predicted = method_results[qid]     # 해당 쿼리의 예측 랭킹
        p,r,rr,ap = compute_metrics(predicted,relevant_dict,k)  # 지표 계산
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)

    return{
        'P@k':np.mean(prec_list),
        'R@k':np.mean(rec_list),
        'MRR':np.mean(rr_list),
        'MAP':np.mean(ap_list)
    }

## 06.BM25와 Dense 결과의 RRF 결합

> bm25와 dense 검색의 결과가 상이할대 rrf는 빛을 발한다.

RRF(Reciprocal Rank Fusion)는 서로 다른 검색 기법의 결과를 하나로 합쳐 더 안정적이고 정확한 순위를 만들어 내는 간단한 랭크 결합 기법이다.

1. 여러 검색 결과 리스트(예: BM25, Dense)에서 **문서의 순위(rank)** 를 이용해 점수를 계산한다.
2. 각 리스트별로, 문서가 n위에 있을 때 주어지는 점수는
   $\displaystyle \frac{1}{k + n}$
   형태이다. 여기서 k는 보통 60처럼 비교적 큰 상수로, 순위 차이에 따른 점수 차이를 완만하게 만들어 준다.
3. 두 개 이상의 리스트에서 **나온 점수를 모두 더해** 최종 RRF-점수를 구한다.
4. 최종 점수가 높은 문서일수록 “여러 기법에서 꾸준히 상위권” 에 올랐다는 뜻이므로, 가장 관련성이 높다고 판단된다.

예를 들어, 문서 A가

* BM25 결과에서 2위 → 점수 1/(60+2)
* Dense 결과에서 5위 → 점수 1/(60+5)
  를 받았다면,

$$
\text{RRF-점수}(A) = \frac{1}{62} + \frac{1}{65}
$$

가 되어, 두 기법에서 모두 좋은 순위를 받은 문서 A가 최종적으로 높은 순위에 오르게 된다.

이처럼 RRF는 각 검색 기법의 강점을 살려 **서로 다른 결과를 보완**하며, 한쪽에서만 높은 점수를 받은 노이즈 문서를 걸러내고, 여러 기법에서 상위권에 오른 핵심 문서를 잘 찾아준다.

In [9]:
bm25_results = {}   # qid별 BM25 검색 결과 저장 dict
for idx, row in queries_df.iterrows():
    qid = row['query_id']   # 쿼리 ID 추출
    query_text = row['query_text']  # 쿼리 문장
    bm25_results[qid] = bm25_search(query_text, top_k = 5)  # {qid : query_text와 가장 가까운 5개의 doc_id}

bm25_results

{'Q1': ['D1', 'D2', 'D3', 'D4', 'D5'],
 'Q2': ['D13', 'D2', 'D1', 'D3', 'D4'],
 'Q3': ['D2', 'D21', 'D14', 'D11', 'D28'],
 'Q4': ['D4', 'D9', 'D1', 'D30', 'D5'],
 'Q5': ['D5', 'D18', 'D19', 'D23', 'D15'],
 'Q6': ['D6', 'D27', 'D17', 'D25', 'D3'],
 'Q7': ['D27', 'D25', 'D7', 'D14', 'D9'],
 'Q8': ['D8', 'D18', 'D30', 'D12', 'D29'],
 'Q9': ['D9', 'D27', 'D1', 'D2', 'D3'],
 'Q10': ['D10', 'D3', 'D17', 'D9', 'D25'],
 'Q11': ['D11', 'D4', 'D14', 'D28', 'D26'],
 'Q12': ['D12', 'D8', 'D9', 'D27', 'D5'],
 'Q13': ['D13', 'D14', 'D26', 'D2', 'D3'],
 'Q14': ['D14', 'D7', 'D25', 'D27', 'D26'],
 'Q15': ['D9', 'D15', 'D26', 'D14', 'D1'],
 'Q16': ['D16', 'D15', 'D9', 'D29', 'D8'],
 'Q17': ['D17', 'D25', 'D6', 'D3', 'D10'],
 'Q18': ['D18', 'D4', 'D28', 'D5', 'D7'],
 'Q19': ['D19', 'D18', 'D14', 'D24', 'D29'],
 'Q20': ['D20', 'D27', 'D10', 'D3', 'D4'],
 'Q21': ['D21', 'D19', 'D25', 'D2', 'D11'],
 'Q22': ['D22', 'D23', 'D19', 'D18', 'D24'],
 'Q23': ['D23', 'D26', 'D24', 'D22', 'D19'],
 'Q24': ['D24', 'D2

In [10]:
# Vector Store 검색 결과를 쿼리별로 수집

dense_results = {}   # qid별 BM25 검색 결과 저장 dict
for idx, row in queries_df.iterrows():
    qid = row['query_id']   # 쿼리 ID 추출
    query_text = row['query_text']  # 쿼리 문장
     # {qid : query_text와 가장 가까운 5개의 doc_id}
    docs = vector_store.similarity_search(query_text, top_k = 5)  
    dense_results[qid] = [doc.metadata['doc_id']for doc in docs]
dense_results

{'Q1': ['D12', 'D1', 'D13', 'D8'],
 'Q2': ['D13', 'D2', 'D12', 'D24'],
 'Q3': ['D3', 'D17', 'D15', 'D18'],
 'Q4': ['D4', 'D16', 'D26', 'D15'],
 'Q5': ['D5', 'D23', 'D7', 'D14'],
 'Q6': ['D6', 'D14', 'D25', 'D13'],
 'Q7': ['D25', 'D7', 'D29', 'D14'],
 'Q8': ['D8', 'D12', 'D18', 'D30'],
 'Q9': ['D9', 'D3', 'D19', 'D15'],
 'Q10': ['D10', 'D25', 'D4', 'D15'],
 'Q11': ['D11', 'D18', 'D19', 'D12'],
 'Q12': ['D12', 'D8', 'D1', 'D13'],
 'Q13': ['D13', 'D2', 'D19', 'D24'],
 'Q14': ['D14', 'D6', 'D26', 'D25'],
 'Q15': ['D15', 'D14', 'D9', 'D11'],
 'Q16': ['D16', 'D23', 'D9', 'D19'],
 'Q17': ['D17', 'D25', 'D15', 'D10'],
 'Q18': ['D18', 'D4', 'D27', 'D14'],
 'Q19': ['D19', 'D18', 'D27', 'D24'],
 'Q20': ['D20', 'D19', 'D22', 'D24'],
 'Q21': ['D21', 'D23', 'D18', 'D24'],
 'Q22': ['D22', 'D23', 'D24', 'D19'],
 'Q23': ['D23', 'D24', 'D22', 'D20'],
 'Q24': ['D23', 'D24', 'D22', 'D19'],
 'Q25': ['D25', 'D7', 'D17', 'D20'],
 'Q26': ['D26', 'D14', 'D7', 'D27'],
 'Q27': ['D27', 'D29', 'D30', 'D28'],
 'Q28

In [13]:
# BM25 + Dense 결과를 RRF로 결합해서 Hybrid 랭킹 생성

def rrf_rank(bm25_list,dense_list,k=60):
    candidate_scores = {}
    for rank,doc in enumerate(bm25_list,1):
        candidate_scores[doc] = candidate_scores.get(doc,0) +1 /(k+rank)
    candidate_scores = {}
    for rank,doc in enumerate(dense_list,1):
        candidate_scores[doc] = candidate_scores.get(doc,0) +1 /(k+rank)

    ranked = sorted(candidate_scores.items(),key=lambda item:item[1],reverse=True)
    return[doc for doc, _ in ranked]

rrf_results = {}
top_k = 5

for idx,row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    bm25_top20 = bm25_results[qid]
    dense_top20 = dense_results[qid]
    rrf_list = rrf_rank(bm25_top20, dense_top20)
    rrf_results[qid] = rrf_list[:top_k]
rrf_results

{'Q1': ['D12', 'D1', 'D13', 'D8'],
 'Q2': ['D13', 'D2', 'D12', 'D24'],
 'Q3': ['D3', 'D17', 'D15', 'D18'],
 'Q4': ['D4', 'D16', 'D26', 'D15'],
 'Q5': ['D5', 'D23', 'D7', 'D14'],
 'Q6': ['D6', 'D14', 'D25', 'D13'],
 'Q7': ['D25', 'D7', 'D29', 'D14'],
 'Q8': ['D8', 'D12', 'D18', 'D30'],
 'Q9': ['D9', 'D3', 'D19', 'D15'],
 'Q10': ['D10', 'D25', 'D4', 'D15'],
 'Q11': ['D11', 'D18', 'D19', 'D12'],
 'Q12': ['D12', 'D8', 'D1', 'D13'],
 'Q13': ['D13', 'D2', 'D19', 'D24'],
 'Q14': ['D14', 'D6', 'D26', 'D25'],
 'Q15': ['D15', 'D14', 'D9', 'D11'],
 'Q16': ['D16', 'D23', 'D9', 'D19'],
 'Q17': ['D17', 'D25', 'D15', 'D10'],
 'Q18': ['D18', 'D4', 'D27', 'D14'],
 'Q19': ['D19', 'D18', 'D27', 'D24'],
 'Q20': ['D20', 'D19', 'D22', 'D24'],
 'Q21': ['D21', 'D23', 'D18', 'D24'],
 'Q22': ['D22', 'D23', 'D24', 'D19'],
 'Q23': ['D23', 'D24', 'D22', 'D20'],
 'Q24': ['D23', 'D24', 'D22', 'D19'],
 'Q25': ['D25', 'D7', 'D17', 'D20'],
 'Q26': ['D26', 'D14', 'D7', 'D27'],
 'Q27': ['D27', 'D29', 'D30', 'D28'],
 'Q28